# 04 · Hình và bảng cho báo cáo

Gom mọi thứ đã chạy thành sản phẩm nộp. Không huấn luyện gì thêm ở đây.

Người phụ trách: **SV B**.

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

In [ ]:
import pandas as pd

cfg = Config()
df = RunLogger(cfg.log_csv).to_dataframe()
for c in [c for c in df.columns if c.startswith(("test_", "best_val", "n_params",
                                                 "train_time", "focal_", "tversky_"))]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ("epochs", "seed", "best_epoch", "epochs_run", "batch_size", "image_size"):
    df[c] = df[c].astype(int)
df["lr"] = df["lr"].astype(float)

print(f"Tổng số lượt chạy trong log: {len(df)}")
print(df.groupby("epochs").size().to_string())

## Bảng tổng hợp cho báo cáo

**Bắt buộc có cột `epochs` và `lr`.** Log chứa nhiều ngân sách huấn luyện và
nhiều learning rate; thiếu hai cột này thì bảng trộn lẫn các điều kiện khác
nhau và mọi so sánh trở nên vô nghĩa.

In [ ]:
summary = (df[["loss_name", "up_mode", "skip_mode", "epochs", "lr", "seed",
                "n_params", "test_dice", "test_iou", "test_precision",
                "test_recall", "best_epoch", "epochs_run", "train_time_min"]]
           .sort_values(["epochs", "test_dice"], ascending=[True, False]).round(4))
print(summary.to_markdown(index=False))

### Bảng chính: so sánh hàm mất mát ở điều kiện được kiểm soát

Cố định kiến trúc, ngân sách và learning rate. Đây là bảng đưa vào báo cáo.

In [ ]:
main = df[(df.epochs == 120) & (df.lr == 1e-3) & (df.seed == 42)
          & (df.up_mode == "transpose") & (df.skip_mode == "full")]
cols = ["loss_name", "test_dice", "test_iou", "test_precision", "test_recall",
        "best_epoch", "epochs_run"]
print(f"Tìm thấy {len(main)} / {len(LOSS_NAMES)} hàm mất mát")
print(main[cols].sort_values("test_dice", ascending=False).round(4).to_markdown(index=False))

#### Chú thích bắt buộc cho bảng chính

Ô dưới tự dò các lượt bị early stopping cắt sớm và các lượt cùng hàm mất mát
nhưng chạy ở learning rate khác. Sinh chú thích từ log thay vì gõ tay.

In [ ]:
odd = main[main.epochs_run < 0.5 * main.epochs]
if len(odd):
    print("CẢNH BÁO — lượt bị early stopping cắt sớm, PHẢI chú thích trong báo cáo:")
    print(odd[["loss_name", "best_epoch", "epochs_run", "test_dice"]].to_markdown(index=False))

alt = df[(df.epochs == 120) & (df.lr != 1e-3) & (df.up_mode == "transpose")
         & (df.skip_mode == "full")]
if len(alt):
    print("\nLượt ở learning rate khác — đưa vào tiểu mục CHẨN ĐOÁN, không vào bảng chính:")
    print(alt[["loss_name", "lr", "best_epoch", "epochs_run", "test_dice",
               "test_precision", "test_recall"]].round(4).to_markdown(index=False))

## Lưới ảnh so sánh (yêu cầu tối thiểu 10 ảnh)

Đề bài yêu cầu: ảnh gốc / mask thật / dự đoán của 2–3 cấu hình tốt nhất và
1 cấu hình tệ nhất.

In [ ]:
PROTO = Config().to_dict()

def cfg_from_row(row):
    """Dựng lại Config ĐẦY ĐỦ từ một dòng log.

    Bắt buộc phải khôi phục mọi trường, không chỉ vài trường dễ thấy: run_id là
    hash của toàn bộ Config, nên thiếu epochs hay lr là ra tên checkpoint khác
    và load_best sẽ báo FileNotFoundError.
    """
    d = {}
    for k, default in PROTO.items():
        if k not in row or pd.isna(row[k]):
            continue
        v = row[k]
        if isinstance(default, bool):
            d[k] = str(v).strip().lower() in ("true", "1")
        elif default is None:
            d[k] = None if str(v).strip() in ("", "None", "nan") else float(v)
        else:
            d[k] = type(default)(v)
    c = Config(**d)
    assert c.run_id == row["run_id"], f"hash lệch: {c.run_id} != {row['run_id']}"
    return c

Dòng `assert` là chốt chặn: nếu hash dựng lại khớp với hash trong log thì
chắc chắn đang nạp đúng mô hình đã sinh ra con số trong bảng.

In [ ]:
from src.viz import comparison_grid

# Chỉ lấy lượt 120 epoch: model 40 epoch chưa hội tụ, đưa vào lưới ảnh sẽ
# so sánh nhầm giữa "hàm mất mát kém" và "chưa train đủ".
pool = df[(df.epochs == 120) & (df.seed == 42)]
best_rows = pool.nlargest(3, "test_dice")

# Cấu hình "tệ nhất" phải CÓ Ý NGHĨA để so sánh, không lấy lượt hỏng.
# skip_mode="none" là ablation có chủ đích; focal@alpha=0.25 chỉ là lỗi tham số.
worst_row = pool[pool.skip_mode == "none"].nsmallest(1, "test_dice").iloc[0]


def label(row):
    """Nhãn phải đủ để tái lập cấu hình.

    Thiếu skip_mode thì hai dòng bce_dice/transpose (full và half) trùng tên;
    thiếu lr thì người đọc tưởng mọi cột đều ở learning rate chuẩn.
    """
    s = f"{row.loss_name}/{row.up_mode}/{row.skip_mode}"
    return s if row.lr == 1e-3 else f"{s}\nlr={row.lr:g}"


models = {}
for _, row in best_rows.iterrows():
    c = cfg_from_row(row)
    models[label(row)] = load_best(build_model(c), c.ckpt_path, get_device())

# KHÔNG gọi cấu hình này là "tệ nhất": trên nhiều ảnh nó còn thắng các cấu
# hình đầu bảng, vì bỏ skip cũng đồng nghĩa bỏ luôn nhiễu độ phân giải cao mà
# skip mang sang. Nó kém đi ở polyp nhỏ và chi tiết biên. Đó là một sự ĐÁNH ĐỔI,
# không phải một thất bại, và nhãn phải phản ánh đúng điều đó.
c = cfg_from_row(worst_row)
models[f"ablation\n{label(worst_row)}"] = load_best(build_model(c), c.ckpt_path,
                                                     get_device())

print(best_rows[["loss_name", "up_mode", "skip_mode", "lr", "test_dice"]]
      .round(4).to_markdown(index=False))
print("\nTệ nhất:", label(worst_row).replace(chr(10), " "), round(worst_row.test_dice, 4))

In [ ]:
splits = load_splits(cfg.split_dir)
test_ds = KvasirSegDataset(cfg.data_root, splits["test"],
                           SegTransform(cfg.image_size, train=False))

# Chọn 10 ảnh TRẢI ĐỀU theo kích thước polyp thay vì lấy 10 ảnh đầu tiên.
# Nhờ vậy lưới ảnh cho thấy mô hình xử lý polyp nhỏ so với polyp lớn thế nào.
import numpy as np
from PIL import Image

areas = np.array([(np.asarray(Image.open(Path(cfg.data_root) / "masks" / n)
                              .convert("L")) > 127).mean() for n in splits["test"]])
order = np.argsort(areas)
picks = [int(order[int(q * (len(order) - 1))]) for q in np.linspace(0.02, 0.98, 10)]
print("Tỉ lệ polyp của 10 ảnh đã chọn:", np.round(areas[picks], 4).tolist())

In [ ]:
# Tách làm hai hình 5 hàng để vừa khổ giấy báo cáo
for k, half in enumerate([picks[:5], picks[5:]], start=1):
    comparison_grid(test_ds, models, indices=half, device=get_device(),
                    save_path=f"{cfg.fig_dir}/comparison_grid_{k}.png")

## Lưới ảnh gây bất đồng

Lưới trải đều kích thước ở trên minh hoạ phổ dữ liệu, nhưng phần lớn hàng cho
Dice trên 0.9 ở mọi cấu hình nên không phân biệt được gì. Lưới dưới chọn những
ảnh mà các cấu hình **bất đồng nhất** — đó mới là bằng chứng cho chênh lệch
trong bảng ablation.

In [ ]:
import torch
from src.metrics import dice_per_image
from torch.utils.data import DataLoader

per_model = {}
for name, m in models.items():
    scores = []
    for images, masks in DataLoader(test_ds, batch_size=8, shuffle=False):
        with torch.no_grad():
            lg = m(images.to(get_device())).cpu()
        scores += dice_per_image(lg, masks).tolist()
    per_model[name] = np.array(scores)

mat = np.stack(list(per_model.values()))
spread = mat.std(axis=0)
contested = np.argsort(-spread)[:5].tolist()

print("5 ảnh gây bất đồng nhất (chỉ số, độ lệch chuẩn giữa các cấu hình):")
for i in contested:
    print(f"  {i:4d}  std={spread[i]:.3f}  " +
          "  ".join(f"{n.splitlines()[0]}={per_model[n][i]:.3f}" for n in per_model))

In [ ]:
comparison_grid(test_ds, models, indices=contested, device=get_device(),
                save_path=f"{cfg.fig_dir}/comparison_contested.png")

### Dice trung bình theo kích thước polyp

Bảng này trả lời câu hỏi quan trọng nhất của phần phân tích lỗi: mỗi cấu hình
mạnh yếu ở loại polyp nào. Đây là chỗ sự đánh đổi của `skip_mode` lộ ra.

In [ ]:
bins = pd.cut(areas, [0, .05, .15, .35, 1.0],
              labels=["rất nhỏ", "nhỏ", "vừa", "lớn"])
tbl = pd.DataFrame({n: s for n, s in per_model.items()})
tbl.columns = [c.splitlines()[0] for c in tbl.columns]
tbl["nhóm"] = bins
out = tbl.groupby("nhóm", observed=False).agg(["mean", "count"]).round(4)
print(out.to_markdown())

## Ảnh mô hình làm tệ nhất

Nguyên liệu cho mục *Phân tích và thảo luận*. Nhìn kỹ xem chúng có điểm chung
gì: polyp quá nhỏ, biên mờ, có bọt khí, ánh sáng chói, hay nhiều polyp.

In [ ]:
from src.viz import worst_case_indices
from torch.utils.data import DataLoader

best_name = list(models)[0]
loader = DataLoader(test_ds, batch_size=8, shuffle=False)
worst = worst_case_indices(models[best_name], loader, get_device(), k=6)
print("Chỉ số 6 ảnh tệ nhất:", worst)

fig = comparison_grid(test_ds, {best_name: models[best_name]}, indices=worst,
                      device=get_device(),
                      save_path=f"{cfg.fig_dir}/worst_cases.png")

## Danh sách kiểm trước khi nộp

- [ ] `logs/runs.csv` có đủ mọi lượt chạy, kể cả lượt hỏng
- [ ] Mọi số trong báo cáo đều đọc từ CSV, không gõ tay
- [ ] `README.md` chạy lại được từ đầu trên máy sạch
- [ ] `requirements.txt` khớp với môi trường thật
- [ ] Bảng phân công có tỉ lệ đóng góp cộng lại đúng 100%
- [ ] Mục *Khai báo sử dụng công cụ AI* ở cuối báo cáo
- [ ] Hình nào cũng có chú thích và được nhắc tới trong nội dung
- [ ] Mọi bảng đều ghi rõ `epochs` và `lr`, không trộn lẫn các ngân sách
- [ ] Cấu hình "tệ nhất" trong lưới ảnh là ablation có chủ đích, không phải lượt hỏng
- [ ] Có ít nhất một cấu hình chạy 2 seed để phát biểu về ý nghĩa thống kê